# 11 — Histology results aggregation

Mirror of notebook 06 for the LC25000 experiment. Loads per-fold JSON files, builds the final results table, ROC + PR curves, confusion matrices, bar chart, and pairwise Wilcoxon p-values.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from sklearn.metrics import (
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
    average_precision_score,
)

from utils.training import load_fold_results
from utils.metrics import aggregate_folds, per_fold_metric_list
from utils.stats import format_results_table, pairwise_wilcoxon

with open('../configs/histology.yaml') as f:
    cfg = yaml.safe_load(f)

results_dir = Path('..') / cfg['paths']['results']
figures_dir = Path('..') / cfg['paths']['figures']
figures_dir.mkdir(parents=True, exist_ok=True)

MODELS = ['SVM', 'RF', 'KNN', 'ResNet50_pretrained', 'ResNet50_scratch']
all_folds = {name: load_fold_results(results_dir / f'histology_{name}.json') for name in MODELS}

In [ ]:
agg = {name: aggregate_folds([f.metrics_calibrated for f in folds]) for name, folds in all_folds.items()}
table = format_results_table(agg)
table.to_csv(results_dir / 'histology_summary_calibrated.csv')
table

In [ ]:
rows = []
for name, folds in all_folds.items():
    for f in folds:
        rows.append({'model': name, 'fold': f.fold, **f.metrics_calibrated})
per_fold_df = pd.DataFrame(rows)
per_fold_df.to_csv(results_dir / 'histology_per_fold.csv', index=False)
per_fold_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, folds in all_folds.items():
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    p, r, _ = precision_recall_curve(y_true, y_proba)
    axes[1].plot(r, p, label=f'{name} (AP={average_precision_score(y_true, y_proba):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].set_title('ROC — out-of-fold'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('PR — out-of-fold'); axes[1].legend()
plt.tight_layout(); plt.savefig(figures_dir / 'histology_roc_pr.png', dpi=140); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4 * len(MODELS), 4))
for ax, (name, folds) in zip(axes, all_folds.items()):
    y_true = np.concatenate([np.array(f.y_true) for f in folds])
    y_proba = np.concatenate([np.array(f.y_proba) for f in folds])
    thr = float(np.mean([f.threshold_calibrated for f in folds]))
    y_pred = (y_proba >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['benign', 'malignant'], yticklabels=['benign', 'malignant'])
    ax.set_title(f'{name}\nthr={thr:.2f}'); ax.set_xlabel('predicted'); ax.set_ylabel('actual')
plt.tight_layout(); plt.savefig(figures_dir / 'histology_confusion.png', dpi=140); plt.show()

In [ ]:
metrics = ['accuracy', 'sensitivity', 'specificity', 'precision', 'f1', 'balanced_accuracy', 'auc_roc', 'pr_auc']
means = pd.DataFrame({m: {name: agg[name][m]['mean'] for name in MODELS} for m in metrics})
stds = pd.DataFrame({m: {name: agg[name][m]['std'] for name in MODELS} for m in metrics})
ax = means.T.plot(kind='bar', yerr=stds.T, figsize=(14, 5), capsize=2)
ax.set_ylim(0, 1.05); ax.set_ylabel('score'); ax.set_title('LC25000 — model comparison (5-fold CV, calibrated)')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig(figures_dir / 'histology_bars.png', dpi=140); plt.show()

In [ ]:
auc_by_model = {name: per_fold_metric_list([f.metrics_calibrated for f in folds], 'auc_roc')
                for name, folds in all_folds.items()}
p_matrix = pairwise_wilcoxon(auc_by_model)
p_matrix.to_csv(results_dir / 'histology_wilcoxon_auc.csv')
p_matrix